In [1]:
from typing import List

In [2]:
import tensorflow as tf

# List available hardware
devices: List = tf.config.list_physical_devices()
print(f"Devices found: {devices}")

Devices found: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
gpu_devices: List = tf.config.list_physical_devices("GPU")

if gpu_devices:
    print(f"GPU devices found: {gpu_devices}")

    for each_gpu_device in gpu_devices:
        tf.config.experimental.set_memory_growth(each_gpu_device, True)

else:
    print("Running on Local CPU")

GPU devices found: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Remarks: How to Fetch the Dataset

We will use `Indonesian Food Dataset` from [Kaggle](https://www.kaggle.com/datasets/rizkyyk/dataset-food-classification/data) for this project

### Dataset Description

Messr Rizky Yanuar curated a dataset of 6500 image datasets of Indonesian foods. The dataset contains 13 food categories, each with ~500 images. These categories include:
* Ayam Goreng (Fried Chicken)
* Burger
* French Fries
* Gado-Gado (Indonesian Salad)
* Ikan Goreng (Fried Fish)
* Mie Goreng (Fried Noodles)
* Nasi Goreng (Fried Rice)
* Nasi Padang (Padang Rice)
* Pizza
* Rawon (Beef Soup)
* Rendang (Spiced Meat)
* Sate (Satay)
* Soto Ayam (Chicken Soup)


### Authentication

Authentication is needed to download this dataset from Kaggle

First, you will need a Kaggle account. You can sign up [here](https://www.kaggle.com/account/login)

After login, you can download your Kaggle API token at [here](https://www.kaggle.com/settings) by clicking on the "Generate new Token" button under the "API" section.

You have several options to authenticate:

Option 1: kagglehub.login()

```Python
import kagglehub

kagglehub.login()
```
This will prompt you to enter your Kaggle API token.

Option 2: Environment Variable

```Bash
export KAGGLE_API_TOKEN=xxxxxxxxxxxxxx # Coped from the settings UI
```

Option 3: API Token File

Store your Kaggle API token obtained from your [setting page](https://www.kaggle.com/settings) to a file located in `~/.kaggle/access_token`

Option 4: Google Colab Secret

If you are running this notebook from Google Cloud, then you can store your API token in a Colab secret named `KAGGLE_API_TOKEN`.

Instructions on adding secrets in Colab can be found [here](https://www.googlecloudcommunity.com/gc/Cloud-Hub/How-do-I-add-secrets-in-Google-Colab-Enterprise/m-p/784866)

Option 5: Legacy API credentials file

From your [Kaggle Account Setting Page](https://www.kaggle.com/settings), under the "Legacy API Credentials", click on the "Create Legeacy API Key" button to generate a `kaggle.json` file and store it at `~/.kaggle/kaggle.json`

In [4]:
# Authenticate Kaggle

import os
from pathlib import Path


import kagglehub

is_authenticated: bool = False

api_key_path: Path = Path("~/.kaggle/kaggle.json").expanduser()
is_api_key_exists: bool = True if api_key_path.exists() else False

api_token_path: Path = Path("~/.kaggle/access_token").expanduser()
is_api_token_exists: bool = True if api_token_path.exists() else False

## Check if running in Colab
def is_colab() -> bool:
    """
    Small utility function to check if this notebook is running on Colab
    :return: Boolean indication if this script is running on Colab
    :rtype bool
    """
    try:
        import google.colab
        return True
    except ImportError:
        return False

is_colab_secret_exists: bool = False
if is_colab():
    from google.colab import userdata

    try:
        is_colab_secret_exists = True if userdata.get("KAGGLE_API_TOKEN") else False
    except userdata.SecretNotFoundError:
        is_colab_secret_exists = False

is_token_env_exists: bool = True if os.getenv("KAGGLE_API_TOKEN") else False

is_token_via_login: bool = False
if all([
    not is_api_key_exists
    , not is_api_token_exists
    , not is_colab_secret_exists
    , not is_token_env_exists
]):
    # if everything above fails, then login interactively
    # this will ask you to input the Kaggle API Token from a UI
    kagglehub.login()
    is_token_via_login = True


if any([
    is_api_key_exists
    , is_api_token_exists
    , is_token_env_exists
    , is_colab_secret_exists
    , is_token_via_login
]):
    is_authenticated = True
    print("Kagglehub has been authenticated")

else:
    raise Exception(" 🛑 Kagglehub fails to authenticate")

Kagglehub has been authenticated


## Remarks: Download the Dataset

In this section, we will download the dataset from Kaggle.

We want to detect if any `.jpg` files already present under `dataset/` folder.

If not, then we: (1) would download the dataset from Kaggle, (2) copy the downloaded data from Kaggle's `cache/` folder into the project's `dataset/` folder. If `.jpg` files already present in the `dataset/` folder, then we will simply load the data from the `dataset/` folder.

This way, we can avoid downloading the dataset every time we run this notebook.

In [5]:
# Download Data Smartly
# If the data is not yet downloaded, we will first download the data from Kaggle to `dataset` folder, and then load them to pandas
# Else, we will load the data from the `dataset` folder directly to pandas

from pathlib import Path
import shutil

import kagglehub

kaggle_dataset_handler_str: str = "rizkyyk/dataset-food-classification"

project_root: Path = Path.cwd().expanduser()
dataset_folder: Path = project_root / "dataset"

dataset_folder.mkdir(parents=True, exist_ok=True)

available_files = sorted(each_found_file.name for each_found_file in dataset_folder.glob("**/*.jpg"))

if available_files:
    print("Dataset already exists locally. Using files from `dataset` folder.")
    print(f"Count of available files: {len(available_files)}")
else:
    downloaded_path_str: str = kagglehub.dataset_download(
        handle=kaggle_dataset_handler_str
    )
    downloaded_path: Path = Path(downloaded_path_str)
    dataset_gambar_path: Path = downloaded_path / "dataset_gambar"

    shutil.copytree(
        src = dataset_gambar_path
        , dst = dataset_folder
        , dirs_exist_ok = True
    )

    available_files = sorted(each_found_file.name for each_found_file in dataset_folder.glob("**/*.jpg"))

    print(f"Count of available files: {len(available_files)}")

Dataset already exists locally. Using files from `dataset` folder.
Count of available files: 6490
